# **Sample Test Code for environment compatibility**

In [ ]:
import torch
import torch.nn.functional as F
from torch.distributions import Normal, kl_divergence

torch.manual_seed(0)

# Simulate encoder output for a batch of 4 samples
mu    = torch.tensor([0.5, -1.0, 0.2, 0.8])
logvar = torch.tensor([-0.5, 0.0, -1.0, 0.3])
sigma  = torch.exp(0.5 * logvar)

# Reparameterize
eps = torch.randn_like(mu)
z   = mu + sigma * eps

# --- KL divergence (closed form) ---
kl_closed = -0.5 * torch.sum(1 + logvar - mu**2 - torch.exp(logvar))

# --- KL divergence (via torch.distributions, for verification) ---
q = Normal(mu, sigma)
p = Normal(torch.zeros_like(mu), torch.ones_like(mu))
kl_lib = torch.sum(kl_divergence(q, p))

print(f"KL (closed form): {kl_closed:.4f}")
print(f"KL (torch lib):   {kl_lib:.4f}")
# These should match (within floating point error)

# --- Fake reconstruction loss (MSE as a stand-in) ---
x_original    = torch.randn(4, 8)
x_reconstructed = torch.randn(4, 8)   # replace with model output in the assignment
recon_loss = F.mse_loss(x_reconstructed, x_original, reduction='sum')

elbo = recon_loss + kl_closed
print(f"\nReconstruction loss: {recon_loss:.4f}")
print(f"ELBO (loss):         {elbo:.4f}")

KL (closed form): 1.2271
KL (torch lib):   1.2271

Reconstruction loss: 65.3049
ELBO (loss):         66.5320


# **Part 1 — Closed-Form KL, From Scratch**

In [ ]:
def kl_divergence_gaussian(mu: torch.Tensor, logvar: torch.Tensor) -> torch.Tensor:
    """
    Closed-form KL divergence between N(mu, exp(logvar)) and N(0, 1).
    Returns a scalar (summed over all dimensions and batch).
    """
    # YOUR CODE HERE
    kl = 0.5 * (torch.exp(logvar) + mu**2 - 1 - logvar)
    # torch.exp(logvar) = variance ( also equals std.Dev.**2)
    # and then using famous ormula for kl divergence  given in notes also.


    return kl.sum()
    # summation over all the kl values calculated for mu of size (32,16)

    pass

# Test it
mu     = torch.randn(32, 16)   # batch of 32, latent dim 16
logvar = torch.randn(32, 16)
sigma  = torch.exp(0.5 * logvar)

your_kl = kl_divergence_gaussian(mu, logvar)

q = Normal(mu, sigma)
p = Normal(torch.zeros_like(mu), torch.ones_like(sigma))
lib_kl  = kl_divergence(q, p).sum()

print(f"Your KL:   {your_kl:.4f}")
print(f"Torch KL:  {lib_kl:.4f}")
assert torch.isclose(your_kl, lib_kl, atol=1e-4), "KL values don't match!"
print("✅ Part 1 passed")

Your KL:   415.6991
Torch KL:  415.6991
✅ Part 1 passed


# **Part 2 — Reparameterization From Scratch**

In [ ]:
def reparameterize(mu: torch.Tensor, logvar: torch.Tensor) -> torch.Tensor:
    """
    Sample z using the reparameterization trick.
    z = mu + std * eps, where eps ~ N(0, I)
    """
    # YOUR CODE HERE
    std = torch.exp(0.5 * logvar)
    eps = torch.randn_like(std)
    z = mu + std * eps
    return z

    pass

# Verify: the mean of many samples should be close to mu
mu_test     = torch.tensor([2.0, -1.0])
logvar_test = torch.tensor([0.0,  0.5])

samples = torch.stack([reparameterize(mu_test, logvar_test) for _ in range(10000)])
print(f"Target mu:       {mu_test.tolist()}")
print(f"Sample mean:     {samples.mean(0).tolist()}")
print(f"Target std:      {torch.exp(0.5 * logvar_test).tolist()}")
print(f"Sample std:      {samples.std(0).tolist()}")
# Means and stds should be close (within ~0.05)
print("✅ Part 2 passed")

Target mu:       [2.0, -1.0]
Sample mean:     [2.0186612606048584, -1.0111300945281982]
Target std:      [1.0, 1.2840254306793213]
Sample std:      [1.0084105730056763, 1.2794467210769653]
✅ Part 2 passed


# **Full ELBO Loss Function ( Reconstruction Loss = Binary Cross Entropy)**

In [ ]:
def elbo_loss(x: torch.Tensor,
              x_recon: torch.Tensor,
              mu: torch.Tensor,
              logvar: torch.Tensor) -> torch.Tensor:
    """
    ELBO loss = Reconstruction loss + KL divergence
    - Reconstruction: BCE between x_recon and x, summed over pixels and batch
    - KL: closed-form KL between N(mu, exp(logvar)) and N(0, I)
    Returns a scalar.
    """
    # YOUR CODE HERE
      # 1. Reconstruction loss
    recon_loss = F.binary_cross_entropy(
        x_recon,
        x,
        reduction="sum"
    )

    # 2. KL divergence
    kl_loss = 0.5 * torch.sum(torch.exp(logvar) + mu**2 - 1 - logvar)

    # 3. Total loss
    loss = recon_loss + kl_loss

    return loss
    pass

# Quick smoke test
batch, dim, latent = 16, 784, 32
x      = torch.rand(batch, dim)
x_recon = torch.sigmoid(torch.randn(batch, dim))
mu     = torch.randn(batch, latent)
logvar = torch.randn(batch, latent)

loss = elbo_loss(x, x_recon, mu, logvar)
print(f"ELBO loss (should be a positive scalar): {loss.item():.4f}")
assert loss.ndim == 0, "Loss must be a scalar!"
print("✅ Part 3 passed")

ELBO loss (should be a positive scalar): 10562.2383
✅ Part 3 passed


# **If mu , logvar = 0 then**

In [ ]:
# Running Elbo Loss with mu and logvar as all zeroes
def elbo_loss(x: torch.Tensor,
              x_recon: torch.Tensor,
              mu: torch.Tensor,
              logvar: torch.Tensor) -> torch.Tensor:
    """
    ELBO loss = Reconstruction loss + KL divergence
    - Reconstruction: BCE between x_recon and x, summed over pixels and batch
    - KL: closed-form KL between N(mu, exp(logvar)) and N(0, I)
    Returns a scalar.
    """
    # YOUR CODE HERE
      # 1. Reconstruction loss
    recon_loss = F.binary_cross_entropy(
        x_recon,
        x,
        reduction="sum"
    )

    # 2. KL divergence
    kl_loss = 0.5 * torch.sum(torch.exp(logvar) + mu**2 - 1 - logvar)

    # 3. Total loss
    loss = recon_loss + kl_loss

    return loss
    pass

    # Quick smoke test
batch, dim, latent = 16, 784, 32
x      = torch.rand(batch, dim)
x_recon = torch.sigmoid(torch.randn(batch, dim))
mu     = torch.zeros(batch, latent)
logvar = torch.zeros(batch, latent)

loss = elbo_loss(x, x_recon, mu, logvar)
print(f"ELBO loss (should be a positive scalar): {loss.item():.4f}")

ELBO loss (should be a positive scalar): 10059.5088


# **Part 4 — Analysis & Reflection-**

Answer the following

1.Why can't we just maximise log p(x) directly? What makes it intractable, and what role does the encoder q(z|x) play in getting around this?

2.What would happen if you removed the KL term from the ELBO loss? What behaviour would you expect from the encoder during training?

3.Why does the reparameterization trick work? Where exactly does the gradient flow, and what stops it from flowing without the trick?

4.Run your elbo_loss with mu all zeros and logvar all zeros. What is the KL term? Does that match the closed-form formula? Explain why.

# **Answers-**
1. We cannot maximize p(x) directly because we would have to check every possible hidden code z, which is not feasible because computation takes time and hence makes it intractable, The encoder q(z | x) acts like a smart guesser that tells us where the important latent codes can be.

2. If we remove the KL term, the VAE only tries to reconstruct the input as accurately as possible. As a result, the model behaves like a normal autoencoder but also latent space becomes messy,which makes it hard to generate new meaningful samples by randomly picking latent vectors.

3. Using reparametrization and epsilon , we break the z as addition amd multiplication of the terms and gradient can flow backwards, Without the reparameterization trick, we would sample z directly from a distribution. Since sampling is a random operation, the computer cannot calculate gradients through it, so the encoder would not be able to learn properly.

4. The KL term comes out to be zero which indeed matches the closed form formula and it is quite intuitive beccause if mu and logvar are zeroes then it matches with previous distribution and their should be zero divergence .
